# DBA-Fusion (RA-L 2024) -- learned/hybrid VIO branch of vio-bench

Runs DBA-Fusion in **monocular VIO mode (no GNSS / odometry)** on an EuRoC/ASL-format sequence, converts the
result to TUM (`results/dbafusion/<dataset>/<seq>/traj_tum.txt`, IMU/body pose, seconds), evaluates with
evo (APE: SE(3) alignment, no scale; RPE: delta in meters) and logs everything to ClearML.

Run top to bottom (Runtime -> GPU: T4 / L4 / A100). Every cell is idempotent: after a disconnect just re-run from the top;
the slow builds are restored from Google Drive (`vio-bench/cache/dbafusion`).
Prerequisites: Colab Secrets `CLEARML_API_ACCESS_KEY`, `CLEARML_API_SECRET_KEY`, `CLEARML_API_HOST`, `CLEARML_WEB_HOST`, `CLEARML_FILES_HOST`
(notebook access ON), and data on Drive at `MyDrive/vio-bench/data/<dataset>/<seq>/asl/mav0/...` + `gt_tum.txt`
(or tick `DOWNLOAD_EUROC`). See `methods/dbafusion/README.md`.

## 0. Parameters

In [ ]:
# ------------------------------- EDIT ME -------------------------------
DATASET = "euroc"                 # -> configs/<DATASET>.yaml and data/<DATASET>/<SEQ>
SEQ = "MH_01_easy"
CONFIG_NAME = None                # None -> f"{DATASET}.yaml"; or e.g. "my_robot.yaml"
OVERRIDES = {}                    # config overrides, e.g. {"dbaf.max_factors": 36, "stride": 3}
MAX_FRAMES = None                 # e.g. 100 for a quick smoke test (does NOT write the final result unless FORCE_RERUN)
RPE_DELTA_M = 1.0                 # RPE delta in meters
FORCE_RERUN = False               # True: ignore existing traj_tum.txt on Drive
DOWNLOAD_EUROC = False            # True: download EuRoC <SEQ> (zip) into Drive (sanity-check convenience)
CLEARML_ENABLED = True

# Where the helper scripts + configs (methods/dbafusion) come from. Order: UPLOADED_DIR, Drive copy, git clone.
REPO_URL = "https://github.com/<USER>/vio-bench.git"   # PLACEHOLDER: set after pushing the repo
REPO_BRANCH = "main"
UPLOADED_DIR = "/content/dbafusion"                    # or upload the folder methods/dbafusion here
DRIVE_ROOT = "/content/drive/MyDrive/vio-bench"

# Pins (do not change unless you know why)
DBAF_REPO = "https://github.com/GREAT-WHU/DBA-Fusion.git"
DBAF_COMMIT = "2c658a3be747247ed0b17af4aef42831ed5e7310"     # 2025-03-11 (main HEAD when pinned); lietorch submodule 0fa9ce8
GTSAM_REPO = "https://github.com/yuxuanzhou97/gtsam.git"      # DBA-Fusion's custom GTSAM fork
GTSAM_COMMIT = "c93ea5ba01dec1bf909ba9392fbe84975cb75a82"     # 2024-02-15 'Update README' = the version released with DBA-Fusion
WEIGHTS_GDRIVE_ID = "1PpqVt1H4maBa_GbPJp4NwxRsd9jk-elh"       # droid.pth (DROID-SLAM TartanAir weights), from DBA-Fusion README
# -----------------------------------------------------------------------

## 1. GPU check

In [ ]:
import subprocess, sys, os, json, time, shutil, platform
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No CUDA GPU: Runtime -> Change runtime type -> T4/L4/A100"
GPU_NAME = torch.cuda.get_device_name(0)
CAP = "%d.%d" % torch.cuda.get_device_capability(0)
PYV = "%d.%d" % sys.version_info[:2]
GPU_MEM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU {GPU_NAME} sm_{CAP.replace('.','')}  {GPU_MEM_GIB:.1f} GiB | python {PYV} | torch {torch.__version__} (CUDA {torch.version.cuda})")
print(subprocess.run("nvcc --version | tail -2", shell=True, capture_output=True, text=True).stdout)
if GPU_MEM_GIB < 11 and "dbaf.max_factors" not in OVERRIDES:
    print("NOTE: <11 GiB GPU; DBA-Fusion default (max_factors=48) needs ~10 GB. Consider OVERRIDES={'dbaf.max_factors': 36}.")

## 2. Google Drive + helper scripts

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CACHE = f"{DRIVE_ROOT}/cache/dbafusion"; os.makedirs(CACHE, exist_ok=True)

METHOD_DIR = None
for cand in (UPLOADED_DIR, f"{DRIVE_ROOT}/code/dbafusion"):
    if os.path.isfile(f"{cand}/scripts/run_dbafusion.py"):
        METHOD_DIR = cand; break
if METHOD_DIR is None:
    if "<USER>" in REPO_URL:
        raise RuntimeError("Helper scripts not found. Either upload the folder methods/dbafusion to " + UPLOADED_DIR +
                           " (or to Drive: vio-bench/code/dbafusion), or set REPO_URL to your pushed repo.")
    if not os.path.isdir("/content/vio-bench-repo/.git"):
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, "/content/vio-bench-repo"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/vio-bench-repo", "pull", "-q"], check=True)
    METHOD_DIR = "/content/vio-bench-repo/methods/dbafusion"
SCRIPTS = f"{METHOD_DIR}/scripts"
CONFIG = f"{METHOD_DIR}/configs/{CONFIG_NAME or DATASET + '.yaml'}"
assert os.path.isfile(CONFIG), CONFIG
print("method dir:", METHOD_DIR, "\nconfig:", CONFIG)

## 3. System + Python dependencies (torch is NOT touched: we build against Colab's torch)

In [ ]:
%%bash
set -e
apt-get -qq update >/dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y build-essential cmake ninja-build libboost-all-dev libeigen3-dev unzip wget rsync >/dev/null
pip -q install evo clearml gdown pyyaml pyparsing setuptools wheel "pybind11>=2.13,<3" scipy
cmake --version | head -1

## 4. DBA-Fusion checkout (pinned) 
No numpy-quaternion / torch_scatter / open3d / h5py needed: our driver does not use them; `torch_scatter` is replaced by a pure-torch shim (`scripts/shims`).

In [ ]:
DBAF = "/content/DBA-Fusion"
def sh(cmd, env=None, every=25):
    """Run a shell command, stream a thinned-out copy of its output (Colab does not show child fds), raise on failure."""
    print("$", cmd)
    p = subprocess.Popen(["/bin/bash", "-c", "set -o pipefail; " + cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    tail = []
    for i, line in enumerate(p.stdout):
        tail = (tail + [line])[-60:]
        if i % every == 0 or "rror" in line: print(line, end="", flush=True)
    p.wait()
    if p.returncode:
        print("".join(tail)); raise RuntimeError(f"failed (exit {p.returncode}): {cmd}")
if not os.path.isdir(f"{DBAF}/.git"):
    sh(f"git clone {DBAF_REPO} {DBAF}")
sh(f"cd {DBAF} && git fetch -q origin && git checkout -q {DBAF_COMMIT} && git submodule update --init --recursive")
print(subprocess.run(f"cd {DBAF} && git rev-parse HEAD && git submodule status", shell=True, capture_output=True, text=True).stdout)

## 5. Custom GTSAM (python wrapper, fork required)
DBA-Fusion needs `BA2GTSAM`, `VelFactor`, `marginalizeOut`, `CombinedImuFactor.evaluateErrorCustom` ... that only exist in the author's GTSAM fork. 
First run builds it from source (~20-45 min on Colab's 2 vCPUs) and caches a tarball on Drive; later sessions restore it in seconds.

In [ ]:
GTSAM_TAR = f"{CACHE}/gtsam-{GTSAM_COMMIT[:8]}-py{PYV}.tar.gz"

def gtsam_ok():
    r = subprocess.run([sys.executable, "-c",
        "import gtsam; assert all(hasattr(gtsam, n) for n in ('BA2GTSAM','VelFactor','marginalizeOut')); print('gtsam custom OK', gtsam.__file__)"],
        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:]); return r.returncode == 0

def build_gtsam(use_system_pybind):
    src = "/content/gtsam"
    if not os.path.isdir(f"{src}/.git"): sh(f"git clone {GTSAM_REPO} {src}")
    sh(f"cd {src} && git checkout -q {GTSAM_COMMIT}")
    sh(f"rm -rf {src}/build && mkdir {src}/build")
    pb = subprocess.run([sys.executable, "-m", "pybind11", "--cmakedir"], capture_output=True, text=True).stdout.strip()
    flags = (f"-DCMAKE_BUILD_TYPE=Release -DGTSAM_BUILD_PYTHON=ON -DGTSAM_PYTHON_VERSION={PYV} -DPYTHON_EXECUTABLE={sys.executable} "
             "-DGTSAM_BUILD_TESTS=OFF -DGTSAM_BUILD_EXAMPLES_ALWAYS=OFF -DGTSAM_BUILD_TIMING_ALWAYS=OFF "
             "-DGTSAM_BUILD_UNSTABLE=OFF -DGTSAM_UNSTABLE_BUILD_PYTHON=OFF -DGTSAM_WITH_TBB=OFF "
             "-DGTSAM_BUILD_WITH_MARCH_NATIVE=OFF -DCMAKE_POLICY_VERSION_MINIMUM=3.5 -DCMAKE_INSTALL_PREFIX=/usr/local")
    if use_system_pybind: flags += f" -Dpybind11_DIR={pb}"
    sh(f"cd {src}/build && cmake .. {flags}", every=5)
    sh(f"cd {src}/build && make -j$(nproc) && make install && ldconfig && make python-install", every=40)

if not gtsam_ok():
    if os.path.exists(GTSAM_TAR):
        sh(f"python {SCRIPTS}/gtsam_cache.py unpack {GTSAM_TAR}")
    if not gtsam_ok():
        t0 = time.time()
        try:
            build_gtsam(use_system_pybind=True)
        except RuntimeError as e:
            print("build with system pybind11 failed -> retrying with GTSAM's bundled pybind11:", e)
            build_gtsam(use_system_pybind=False)
        print(f"gtsam build took {(time.time()-t0)/60:.1f} min")
        assert gtsam_ok(), "gtsam built but custom API missing"
        sh(f"python {SCRIPTS}/gtsam_cache.py pack {GTSAM_TAR}")

## 6. Build `lietorch` + `droid_backends` against Colab's torch (wheels cached on Drive)
Upstream `setup.py` only emits SASS up to sm_86 (no PTX), so it would not run on L4 (sm_89) etc.; `patch_build.py` regenerates the setup files for *this* GPU's
compute capability and replaces deprecated `Tensor.type()` in `AT_DISPATCH_*`. Wheels are keyed by python/torch/CUDA/arch.

In [ ]:
KEY = f"py{PYV}_torch{torch.__version__.replace('+','_')}_sm{CAP.replace('.','')}"
WHEELS = f"{CACHE}/wheels/{KEY}"; os.makedirs(WHEELS, exist_ok=True)

def ext_ok():
    code_ = ("import torch, lietorch, droid_backends; from lietorch import SE3; "
             "x=SE3.Random(8,device='cuda'); e=(x*x.inv()).log().abs().max().item(); assert e<1e-4, e; print('lietorch+droid_backends OK', e)")
    r = subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-600:]); return r.returncode == 0

def build_ext(eigen_dir=None):
    env = dict(os.environ, MAX_JOBS="2", TORCH_CUDA_ARCH_LIST=CAP, CUDA_HOME=os.environ.get("CUDA_HOME", "/usr/local/cuda"))
    ed = f"--eigen_dir {eigen_dir}" if eigen_dir else ""
    sh(f"python {SCRIPTS}/patch_build.py --root {DBAF} --arch {CAP} {ed}")
    for name in ("setup_lietorch.py", "setup_droid_backends.py"):
        sh(f"cd {DBAF} && python {name} bdist_wheel -d {WHEELS}", env=env, every=15)

have = lambda p: any(f.startswith(p) for f in os.listdir(WHEELS))
if not ext_ok():
    if not (have("lietorch") and have("droid_backends")):
        t0 = time.time()
        try:
            build_ext()
        except Exception as e:
            print("default Eigen build failed, retrying with system Eigen 3.4:", e)
            build_ext("/usr/include/eigen3")
        print(f"extension build took {(time.time()-t0)/60:.1f} min")
    sh(f"pip -q install --force-reinstall --no-deps {WHEELS}/*.whl")
    assert ext_ok()
print("wheel cache:", os.listdir(WHEELS))

## 7. Weights + import smoke test

In [ ]:
WEIGHTS = f"{CACHE}/droid.pth"
if not os.path.exists(WEIGHTS):
    sh(f"gdown --id {WEIGHTS_GDRIVE_ID} -O {WEIGHTS}.part && mv {WEIGHTS}.part {WEIGHTS}")
print("weights:", WEIGHTS, f"{os.path.getsize(WEIGHTS)/2**20:.0f} MiB")
smoke = f"""
import sys; sys.path.insert(0, '{DBAF}/dbaf'); sys.path.append('{SCRIPTS}/shims')
import torch, matplotlib; matplotlib.use('Agg')
import gtsam, lietorch, droid_backends
from dbaf import DBAFusion
sd = torch.load('{WEIGHTS}', map_location='cpu'); assert any('update.weight.2.weight' in k for k in sd), 'unexpected checkpoint'
print('imports OK, checkpoint keys:', len(sd))
"""
r = subprocess.run([sys.executable, "-c", smoke], capture_output=True, text=True); print(r.stdout, r.stderr[-1500:])
assert r.returncode == 0

## 8. Data
Expected: `MyDrive/vio-bench/data/<dataset>/<seq>/asl/mav0/{cam0,imu0}/...` and `gt_tum.txt` (CONTRACT.md). The sequence is staged to local disk (Drive FUSE is slow for thousands of PNGs).

In [ ]:
DATA_DIR = f"{DRIVE_ROOT}/data/{DATASET}/{SEQ}"
if DOWNLOAD_EUROC:
    assert DATASET == "euroc"
    sh(f"python {SCRIPTS}/fetch_euroc.py --data_root {DRIVE_ROOT}/data --seq {SEQ}")
LOCAL = f"/content/work/data/{DATASET}/{SEQ}/asl"
os.makedirs(f"{LOCAL}/mav0", exist_ok=True)
for sub in ("cam0", "imu0"):
    sh(f"rsync -a {DATA_DIR}/asl/mav0/{sub} {LOCAL}/mav0/")
GT = f"{DATA_DIR}/gt_tum.txt"
print("GT present:", os.path.exists(GT))
sh(f"python {SCRIPTS}/asl_to_dbafusion.py --asl_dir {LOCAL} --config {CONFIG} --check")

## 9. ClearML

In [ ]:
import yaml
sys.path.insert(0, SCRIPTS)
import asl_to_dbafusion as A
CFG = A.load_config(CONFIG, OVERRIDES)

task = None
if CLEARML_ENABLED:
    try:
        from google.colab import userdata
        for k in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY", "CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST"):
            os.environ[k] = userdata.get(k)
        from clearml import Task
        task = Task.init(project_name="vio-bench", task_name=f"dbafusion/{DATASET}/{SEQ}", reuse_last_task_id=False,
                         tags=["dbafusion", DATASET, SEQ], auto_connect_frameworks={"matplotlib": False, "pytorch": False})
        task.connect(CFG, name="config")
        task.connect({"dataset": DATASET, "seq": SEQ, "overrides": OVERRIDES, "max_frames": MAX_FRAMES, "rpe_delta_m": RPE_DELTA_M,
                      "dbaf_commit": DBAF_COMMIT, "gtsam_commit": GTSAM_COMMIT, "gpu": GPU_NAME, "compute_capability": CAP,
                      "torch": torch.__version__, "cuda": torch.version.cuda, "python": platform.python_version()}, name="run")
        print("ClearML task:", task.get_output_log_web_page())
    except Exception as e:
        print("!! ClearML disabled:", repr(e)); task = None

## 10. Run DBA-Fusion (VIO, no GNSS)

In [ ]:
RES = f"{DRIVE_ROOT}/results/dbafusion/{DATASET}/{SEQ}"; os.makedirs(RES, exist_ok=True)
WORK = f"/content/work/{DATASET}/{SEQ}"; os.makedirs(WORK, exist_ok=True)
TRAJ = f"{RES}/traj_tum.txt"
RAW, STATS, LOG = f"{WORK}/raw_result.txt", f"{WORK}/run_stats.json", f"{WORK}/log.txt"
smoke_only = MAX_FRAMES is not None
if os.path.exists(TRAJ) and not FORCE_RERUN and not smoke_only:
    print("traj_tum.txt exists on Drive -> skipping run (FORCE_RERUN=True to redo)")
    STATS_D = json.load(open(f"{RES}/run_stats.json")) if os.path.exists(f"{RES}/run_stats.json") else {}
else:
    cmd = [sys.executable, "-u", f"{SCRIPTS}/run_dbafusion.py", "--dbaf_root", DBAF, "--asl_dir", LOCAL, "--config", CONFIG,
           "--weights", WEIGHTS, "--out", RAW, "--stats", STATS]
    for k, v in OVERRIDES.items(): cmd += ["--set", f"{k}={json.dumps(v)}"]
    if MAX_FRAMES: cmd += ["--max_frames", str(MAX_FRAMES)]
    t0 = time.time()
    with open(LOG, "w") as lf:
        p = subprocess.Popen(cmd, cwd=DBAF, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            lf.write(line); lf.flush()
            if any(s in line for s in ("[adapter]", "[run]", "[stats]", "[done]", "Error", "error", "Traceback", "IMU excitation")): print(line, end="")
        p.wait()
    shutil.copy(LOG, f"{RES}/log.txt" if not smoke_only else f"{RES}/log_smoke.txt")
    if p.returncode:
        print(open(LOG).read()[-4000:]); raise RuntimeError(f"run_dbafusion failed (exit {p.returncode}); full log: {RES}/log.txt")
    STATS_D = json.load(open(STATS)); STATS_D["total_wall_time_incl_startup_s"] = time.time() - t0
    import to_tum
    TRAJ_OUT = TRAJ if not smoke_only else f"{RES}/traj_tum_smoke.txt"
    info = to_tum.convert(RAW, TRAJ_OUT); print("TUM:", info)
    STATS_D.update(info)
    json.dump(STATS_D, open(f"{RES}/run_stats.json" if not smoke_only else f"{RES}/run_stats_smoke.json", "w"), indent=2)
    shutil.copy(RAW, f"{RES}/raw_result.txt")
    shutil.copy(CONFIG, f"{RES}/config_used.yaml"); yaml.safe_dump(CFG, open(f"{RES}/config_effective.yaml", "w"))
    if smoke_only: TRAJ = TRAJ_OUT
print(json.dumps(STATS_D, indent=1))

## 11. Evaluate (evo: APE with SE(3) alignment, no scale; RPE with delta in meters) and report to ClearML

In [ ]:
import eval_traj, matplotlib.pyplot as plt
assert os.path.exists(GT), f"ground truth missing: {GT}"
metrics, figs = eval_traj.evaluate(GT, TRAJ, rpe_delta_m=RPE_DELTA_M)
metrics["runtime"] = STATS_D
json.dump(metrics, open(f"{RES}/eval_metrics{'_smoke' if smoke_only else ''}.json", "w"), indent=2)
for n, f in figs.items(): f.savefig(f"{RES}/{n}{'_smoke' if smoke_only else ''}.png", dpi=130)
print(json.dumps({k: v for k, v in metrics.items() if k != "runtime"}, indent=1))
plt.show()

if task is not None:
    lg = task.get_logger()
    for kind, key in (("APE_translation_m", "ape_trans_m"), ("RPE_translation_m", "rpe_trans_m")):
        if metrics.get(key):
            for s, v in metrics[key].items(): lg.report_scalar(kind, s, float(v), iteration=0)
    for s in ("wall_time_s", "fps", "gpu_max_allocated_gib", "gpu_max_reserved_gib", "frames_processed", "total_wall_time_incl_startup_s"):
        if STATS_D.get(s) is not None: lg.report_scalar("runtime", s, float(STATS_D[s]), iteration=0)
    lg.report_scalar("trajectory", "n_associated_poses", metrics["n_assoc"], iteration=0)
    lg.report_scalar("trajectory", "path_length_m", metrics["path_length_m"], iteration=0)
    for n, f in figs.items(): lg.report_matplotlib_figure(title=n, series=n, figure=f, iteration=0, report_image=True)
    task.upload_artifact("traj_tum", artifact_object=TRAJ)
    task.upload_artifact("eval_metrics", artifact_object=metrics)
    task.upload_artifact("config_effective", artifact_object=CFG)
    for nm, pth in (("log", f"{RES}/log.txt"), ("run_stats", f"{RES}/run_stats.json")):
        if os.path.exists(pth): task.upload_artifact(nm, artifact_object=pth)
    task.flush(wait_for_uploads=True)
    print("ClearML:", task.get_output_log_web_page())

In [ ]:
if task is not None: task.close()
print("results on Drive:", RES, os.listdir(RES))